<img src="https://camo.githubusercontent.com/937931bc39d9211a7d16803fcea8ce621422a635f72934af703468774e30bcd7/68747470733a2f2f73656261737469616e72617363686b612e636f6d2f696d616765732f4c4c4d732d66726f6d2d736372617463682d696d616765732f636830325f636f6d707265737365642f30342e77656270" width="500">

A tokenizer converts raw text into smaller units called tokens, which can then be converted into numbers and processed by a neural network.
______
### Types of Tokenizers

**1. Word Tokenization** || "I love AI" → `["I","love","AI"]` || Simple || Vocabulary explodes || Different forms stored separately (`run`, `running`, `runs`, etc.) || Fails on unseen words || Common before deep learning

**2. Character Tokenization** || "hello" → `['h','e','l','l','o']` || No unknown words || Sequences become very long || Higher attention/computation cost || "Transformers" = 12 character tokens vs 1 word token || Not used in modern LLMs

**3. Subword Tokenization (Modern Standard)** || Stores common word pieces instead of full words or characters || `unhappy` → `["un","happy"]` || `playing` → `["play","ing"]` || Small vocabulary || Handles unseen words || Efficient || Used in modern LLMs like GPT


_____

### Main Families of Modern Tokenizers

**1. BPE (Byte Pair Encoding)** || Used in GPT-2, GPT-3, RoBERTa || Starts with characters (`h e l l o`) || Repeatedly merges most frequent pairs (`h+e→he`, `he+l→hel`) || Vocabulary grows gradually through frequent merges || Historically the most important tokenizer family.

**2. WordPiece** || Used in BERT and many Google models || Similar to BPE || BPE merges based on frequency || WordPiece merges based on likelihood improvement || More statistically driven.

________
#### Byte-Level BPE

Used in GPT || Solves handling of unusual symbols (`₹`, `🔥`, `©`, `汉`) || Represents text as bytes instead of characters || Every character is converted to bytes || Since all text/files can be represented as bytes, anything can be tokenized || Eliminates the unknown token (OOV) problem.

_______
### Vocabulary Size Tradeoff

**Small Vocabulary (~1,000 tokens)** || Small embedding matrix || Words split into many tokens (`unbelievable` → `u n b e l i e v a b l e`) || Longer sequences

**Large Vocabulary (~1,000,000 tokens)** || Shorter sequences || Huge embedding matrix || High memory usage

**Modern LLMs** || Balance both tradeoffs || Typically use **32k–128k vocabulary size**.


In [ ]:
import os
import requests

if not os.path.exists("the-verdict.txt"):
    url = (
        "https://raw.githubusercontent.com/rasbt/"
        "LLMs-from-scratch/main/ch02/01_main-chapter-code/"
        "the-verdict.txt"
    )
    file_path = "the-verdict.txt"

    response = requests.get(url, timeout=30)
    response.raise_for_status()
    with open(file_path, "wb") as f:
        f.write(response.content)

In [ ]:
with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', raw_text)
preprocessed = [item.strip() for item in preprocessed if item.strip()]

all_words = sorted(set(preprocessed))
vocab_size = len(all_words)

vocab = {token:integer for integer,token in enumerate(all_words)}

##SIMPLE HARDCODED TOKENIZER

In [ ]:
import re

In [ ]:
class SimpleTokenizer:
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = { i:s for s,i in vocab.items()}

    def encode(self, text):
        preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text)
        preprocessed = [item.strip() for item in preprocessed if item.strip()]
        preprocessed = [
            item if item in self.str_to_int
            else "<|unk|>" for item in preprocessed
        ]

        ids = [self.str_to_int[s] for s in preprocessed]
        return ids

    def decode(self, ids):
        text = " ".join([self.int_to_str[i] for i in ids])
        # Replace spaces before the specified punctuations
        text = re.sub(r'\s+([,.:;?!"()\'])', r'\1', text)
        return text

Text --> Regex split-->Tokens-->Vocabulary lookup-->Token IDs

In [ ]:
all_words.extend(["<|endoftext|>", "<|unk|>"])
vocab = {token: integer for integer, token in enumerate(all_words)}

tokenizer = SimpleTokenizer(vocab)

text1 = "Hello, do you like tea?"
text2 = "In the sunlit terraces of the palace."

text = " <|endoftext|> ".join((text1, text2))

id= tokenizer.encode(text)
text= tokenizer.decode(id)
print(text)

Some tokenizers use special tokens to help the LLM with additional context

Some of these special tokens are

**1. [BOS]** (beginning of sequence) marks the beginning of text

**2. [EOS]** (end of sequence) marks where the text ends (this is usually used to concatenate multiple unrelated texts, e.g., two different Wikipedia articles or two different books, and so on)

**3. [PAD]** (padding) if we train LLMs with a batch size greater than 1 (we may include multiple texts with different lengths; with the padding token we pad the shorter texts to the longest length so that all texts have an equal length)

**4. [UNK]** to represent words that are not included in the vocabulary

_____
**GPT Special Tokens** || Uses `<|endoftext|>` as the padding token || Padding tokens are ignored via attention masks during training, so the exact padding token is not important

**No `<UNK>` Token in GPT-2** || GPT-2 does not use an unknown-word token (`<UNK>`) || Uses Byte-Pair Encoding (BPE) tokenizer || Unseen words are split into subword units instead of being replaced by `<UNK>` || Eliminates out-of-vocabulary (OOV) word problems.


#BPE TOKENIZER

### Problem BPE Solves

Word tokenization creates very large vocabularies, cannot handle unseen words well, and requires different forms of the same word to be stored separately. Character tokenization solves the unseen-word problem but produces much longer sequences, increasing computational cost. BPE was designed to balance both by keeping the vocabulary manageable while still handling unseen words efficiently and maintaining relatively short sequences.


### Core Idea of BPE

BPE stores **frequent subword pieces** instead of whole words or individual characters. This provides a **smaller vocabulary**, can **handle unseen words**, and produces **shorter sequences than character tokenization**.

_____
## BPE Training Pipeline

1. **Initialize vocabulary with individual characters** || Every word is split into characters.

2. **Count all adjacent token pairs** in the training corpus || Find which pair occurs most frequently.

3. **Merge the most frequent pair** into a new token || e.g., `l + o → lo`.

4. **Replace all occurrences** of that pair in the corpus with the new token.

5. **Repeat count → merge → replace** || Each iteration adds one new token to the vocabulary.

6. **Stop when desired vocabulary size is reached**.

**Example:** `l o w` → `lo w` → `low` || `e r` → `er` || Eventually common subwords like `low`, `er`, `est`, and `new` become vocabulary entries and are used for tokenization.


____
GPT uses BPE to keep the vocabulary small and efficient while still handling unseen words. Instead of storing millions of complete words, GPT uses a vocabulary of about 50,257 subword tokens, greatly reducing memory requirements + No unknown words

### Byte-Level BPE

GPT-2 improved BPE by **tokenizing bytes instead of characters**. Since every character can be represented as bytes, the tokenizer can process **any Unicode character, language, emoji, or code snippet**. This removes the need for an **`<UNK>` token**, as every possible input can be represented and tokenized.

### Updated Training Loop (Byte-Level BPE)

The training process remains the same as standard BPE, but it starts with **bytes as the initial vocabulary instead of characters**. Text is first converted into byte representations, then the tokenizer repeatedly **counts the most frequent adjacent byte pairs, merges them into new tokens, updates the corpus, and repeats** until the target vocabulary size is reached. Over time, frequently occurring byte sequences become subword tokens, allowing the model to efficiently represent text from any language or symbol set.

_____
### Why GPT Doesn't Need `<UNK>`

GPT's **Byte-Level BPE** can always break any input into smaller byte-based tokens. If a word has never been seen before, it is simply split into smaller subword or byte pieces rather than being replaced with an **`<UNK>` token**. As a result, every possible input can be represented, eliminating the out-of-vocabulary problem.

______
### Weaknesses of BPE

**Strange Word Splits** || Words may be split in unnatural ways depending on learned merges || e.g., `understanding → under stand ing` or `underst and ing`

**Non-Linguistic Token Boundaries** || BPE merges based on **frequency**, not grammar or word structure

**Arithmetic Issues** || Numbers may be split into awkward token sequences || e.g., `123456789` may be broken into multiple tokens || One reason LLMs can struggle with exact arithmetic.

____
### Advantages of BPE

**Handles Unseen Words** || New or rare words can be broken into subword pieces instead of becoming unknown tokens

**Smaller Vocabulary** || No need to store every possible word

**Shorter Sequences** || Produces fewer tokens than character tokenization, improving efficiency

**Language Independent** || Especially with **Byte-Level BPE**, which can tokenize any language, symbol, or emoji.


### ASCII vs Unicode

**ASCII** || Designed for English || 128 characters only || e.g., `A→65`, `a→97` || Cannot represent Hindi, Chinese, emojis, currency symbols, etc.

**Unicode** || Universal character standard || Assigns a unique code point to every character across languages and symbols || e.g., `A→U+0041`, `न→U+0928`, `😊→U+1F60A`

---

### Computers Store Bytes, Not Characters

Characters are stored as **bytes**, usually using **UTF-8 encoding**.

**Examples**

* `A` → 1 byte
* `न` → 3 bytes
* `😊` → 4 bytes

---

### What GPT Actually Sees

GPT does **not tokenize characters directly** || Text is first converted into UTF-8 bytes || The tokenizer initially works on these byte values rather than letters or Unicode code points.

---

### Byte-Level BPE Training

**Initial Vocabulary** || Contains only the 256 possible byte values (`0–255`)

**Training Process** || Repeatedly finds the most frequent adjacent byte pairs and merges them into new tokens

**Example** || Frequent byte sequences representing `न` may become one token || Frequent combinations like `न + म` may merge into `नम` || Very common words like `नमस्ते` may eventually become a single token

**Result** || The tokenizer learns tokens ranging from individual bytes to characters, subwords, and even complete words based on frequency.


In [ ]:
!pip install tiktoken

In [ ]:
import importlib
import tiktoken

tokenizer = tiktoken.get_encoding("gpt2")

#DATA LOADER

<img  src= "https://camo.githubusercontent.com/d29cea0621158ecea28778582d1ca8f2fbae94a85808d758cd0930fa2587d88a/68747470733a2f2f73656261737469616e72617363686b612e636f6d2f696d616765732f4c4c4d732d66726f6d2d736372617463682d696d616765732f636830325f636f6d707265737365642f31322e77656270" width= 500>

<img  src= "https://camo.githubusercontent.com/3b88416514e89a7fe6b3043bb8afd6b8f9a6a843269038fc6af5e5cd025b6c99/68747470733a2f2f73656261737469616e72617363686b612e636f6d2f696d616765732f4c4c4d732d66726f6d2d736372617463682d696d616765732f636830325f636f6d707265737365642f31332e776562703f313233" width= 500>

In [ ]:
!pip install torch

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader

class GPTDataset(Dataset):
    def __init__(self, txt, tokenizer, max_length, stride):
        self.input_ids = []
        self.target_ids = []

        # Tokenize the entire text
        token_ids = tokenizer.encode(txt, allowed_special={"<|endoftext|>"})
        assert len(token_ids) > max_length, "Number of tokenized inputs must at least be equal to max_length+1"

        # Use a sliding window to chunk the book into overlapping sequences of max_length
        for i in range(0, len(token_ids) - max_length, stride):
            input_chunk = token_ids[i:i + max_length]
            target_chunk = token_ids[i + 1: i + max_length + 1]
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]

In [ ]:
def create_dataloader(txt, batch_size=4, max_length=256,
                         stride=128, #How much the sliding window moves
                         shuffle=True, drop_last=True,
                         num_workers=0):

    # Initialize the tokenizer
    tokenizer = tiktoken.get_encoding("gpt2")

    # Create dataset
    dataset = GPTDataset(txt, tokenizer, max_length, stride)

    # Create dataloader
    dataloader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=drop_last,
        num_workers=num_workers
    )

    return dataloader

##EMBEDDING


<img src="https://camo.githubusercontent.com/2a77e1836de72cc6a8c806b45873d0418c958f8c3c383b5dac3bad31d221db69/68747470733a2f2f73656261737469616e72617363686b612e636f6d2f696d616765732f4c4c4d732d66726f6d2d736372617463682d696d616765732f636830325f636f6d707265737365642f31392e77656270" width= 500>

### The Problem Embeddings Solve

Token IDs are just **identifiers**, not meaningful numbers. A neural network sees tokens like `cat→5`, `dog→12`, `car→91` as plain integers and may incorrectly infer relationships from their values like treating them as a order . Just like student roll numbers do not indicate intelligence, token IDs contain **no semantic information**.

### Goal of Embeddings

Convert tokens into vectors such that **similar words have similar representations** and unrelated words are farther apart. For example, `cat`, `dog`, `puppy`, and `kitten` should have similar vectors, while `cat` and `airplane` should be very different.

### Embedding

**Definition** || A learned dense vector representation of a token.

**Examples** || `cat → [0.1, 0.9, -0.3]` || `dog → [0.2, 0.8, -0.2]` || `car → [-0.7, 0.1, 0.9]` || These vectors capture semantic similarity between tokens.
_______
### Embedding Matrix

The **Embedding Matrix (`W_E`)** is a learnable table that stores the vector representation of every token in the vocabulary.

**Shape** || `(vocab_size, embedding_dim)` || e.g., `(50000, 768)`

**Structure** || Each row corresponds to one token ID and contains its embedding vector.

**Example** || Token `0 → row 0` || Token `1 → row 1` || ... || Token `49999 → row 49999`

_______
### Embedding Lookup

When a token ID is received, the embedding layer simply **indexes the corresponding row** from the embedding matrix.

**Example** || `token_id = 5` → Output = `W_E[5]`

**Key Point** || No complex computation is performed during lookup; the embedding layer simply retrieves the token's learned vector from the embedding matrix.
__________
### Token Embeddings vs Positional Embeddings

**Token Embedding** || Captures **what the word/token is**

**Problem** || Token embeddings alone do not encode word order || Sentences with the same words but different order would have identical token embeddings

**Position Embedding** || Captures **where the token appears in the sequence**

**Final Input to Transformer** || `Final Embedding = Token Embedding + Position Embedding`

**Result** || Model knows both **what the token is** and **where it occurs** || This combined embedding is fed into the attention layers.


### GPT Training Pipeline

**1. Tokenization** || `"I love AI"` → `[10, 25, 42]`

**2. Create Input-Target Pairs** || Input: `[10, 25]` (`I love`) || Target: `[25, 42]` (`love AI`) || Goal: Predict the next token from previous tokens

**3. Embedding Lookup** || Use embedding matrix `W_E (vocab_size, d_model)` || Lookup `W_E[10]`, `W_E[25]` || Output shape: `(2, 768)`

**4. Add Positional Embeddings** || Retrieve position embeddings from `W_P (max_context, d_model)` || `Final Embedding = Token Embedding + Position Embedding` || Output: `X (2, 768)`

**5. Pass Through Transformer** || `X` → Multi-Head Attention → Feed Forward Network → Repeat for all layers || Output: Hidden States `H (2, 768)`

**6. Output Projection** || Use output matrix `W_out (768, vocab_size)` || `logits = H @ W_out` || Output shape: `(2, 50000)` || One score for every vocabulary token

**7. Softmax** || Convert logits into probability distribution over all vocabulary tokens

**8. Loss Calculation** || Compare predicted probabilities with true next tokens using **Cross-Entropy Loss**

**9. Backpropagation** || Error flows backward: `Loss → Output Layer → Transformer → Embedding Layer`

**10. Parameter Update** || Update transformer weights, output layer weights, and embedding vectors using gradient descent

**Key Insight** || Embeddings start as **random vectors** and gradually become meaningful through millions of training updates.
______________________
### GPT Inference Pipeline

**1. Tokenization** || `"I love"` → `[10, 25]`

**2. Embedding Lookup** || Convert token IDs to embeddings using `W_E` || Output shape: `(2, 768)`

**3. Add Positional Embeddings** || `Token Embedding + Position Embedding`

**4. Pass Through Transformer** || Multi-Head Attention + Feed Forward layers || Output hidden states shape: `(2, 768)`

**5. Output Projection** || `H @ W_out` || Output logits shape: `(2, 50000)`

**6. Softmax** || Convert logits into probabilities over the vocabulary

**7. Select Next Token** || Use **Greedy Decoding (argmax)** or **Sampling (temperature, top-k, top-p)**

**8. Generate Token** || e.g., `AI → token 42`

**9. Append Token** || `[10, 25]` → `[10, 25, 42]`

**10. Repeat** || Feed updated sequence back into the model and generate the next token until a stopping condition is reached.
____________________________
### Weight Tying

**Idea** || Instead of learning separate input and output matrices, GPT reuses the embedding matrix for output projection

**Input Embedding Matrix** || `W_E` shape: `(50000, 768)`

**Output Projection Matrix** || `W_out = W_E^T` shape: `(768, 50000)`

**Definition** || Sharing the embedding and output projection weights is called **Weight Tying**

### Benefits

**Without Weight Tying** || Separate `W_E` and `W_out` matrices || ~76 million parameters

**With Weight Tying** || Only one shared matrix || ~38 million parameters || About half the memory

### Intuition

**Embedding Layer** || `Token → Meaning Vector`

**Output Layer** || `Meaning Vector → Token`

Since these operations are roughly inverse of each other, using the same matrix for both is efficient and conceptually sensible.


In [ ]:
import torch
import torch.nn as nn

vocab_size = 50257      # GPT-2 vocabulary size
context_length = 1024   # maximum sequence length
emb_dim = 768           # GPT-2 Small embedding size

# Trainable token embeddings
tok_emb = nn.Embedding(
    num_embeddings=vocab_size,
    embedding_dim=emb_dim
)

# Trainable positional embeddings
pos_emb = nn.Embedding(
    num_embeddings=context_length,
    embedding_dim=emb_dim
)

### Embedding Layer vs Linear Layer

| Feature            | Embedding Layer (`nn.Embedding`)                                | Linear Layer (`nn.Linear`)                                                    |
| ------------------ | --------------------------------------------------------------- | ----------------------------------------------------------------------------- |
| **Core Operation** | Lookup operation (selects a row using an integer index)         | Matrix multiplication (`xW^T + b`)                                            |
| **Input Format**   | Integer token IDs (e.g., `42`)                                  | Continuous vectors (e.g., `[0.12, 0.88, 0.05]`)                               |
| **Typical Use**    | Beginning of models to convert tokens into dense vectors        | Used throughout the network for transformation, projection, or classification |
| **Computation**    | Direct row lookup from weight matrix                            | Dense matrix multiplication                                                   |
| **Efficiency**     | Faster and memory efficient; avoids unnecessary multiplications | Standard dense computation on continuous values                               |

**Key Insight** || `nn.Embedding` is essentially a learnable lookup table (`token ID → vector`) while `nn.Linear` performs mathematical transformations on vectors.


In [ ]:
import tiktoken
import torch
from torch.utils.data import Dataset, DataLoader


class GPTDatasetV1(Dataset):
    def __init__(self, txt, tokenizer, max_length, stride):
        self.input_ids = []
        self.target_ids = []

        # Tokenize the entire text
        token_ids = tokenizer.encode(txt, allowed_special={"<|endoftext|>"})

        # Use a sliding window to chunk the book into overlapping sequences of max_length
        for i in range(0, len(token_ids) - max_length, stride):
            input_chunk = token_ids[i:i + max_length]
            target_chunk = token_ids[i + 1: i + max_length + 1]
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]


def create_dataloader_v1(txt, batch_size, max_length, stride,
                         shuffle=True, drop_last=True, num_workers=0):
    # Initialize the tokenizer
    tokenizer = tiktoken.get_encoding("gpt2")

    # Create dataset
    dataset = GPTDatasetV1(txt, tokenizer, max_length, stride)

    # Create dataloader
    dataloader = DataLoader(
        dataset, batch_size=batch_size, shuffle=shuffle, drop_last=drop_last, num_workers=num_workers)

    return dataloader


with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

vocab_size = 50257
output_dim = 256
context_length = 1024


token_embedding_layer = torch.nn.Embedding(vocab_size, output_dim)
pos_embedding_layer = torch.nn.Embedding(context_length, output_dim)

batch_size = 8
max_length = 4
dataloader = create_dataloader_v1(
    raw_text,
    batch_size=batch_size,
    max_length=max_length,
    stride=max_length
)

In [ ]:
for batch in dataloader:
    x, y = batch

    token_embeddings = token_embedding_layer(x)
    pos_embeddings = pos_embedding_layer(torch.arange(max_length))

    input_embeddings = token_embeddings + pos_embeddings

    break